# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a walkthrough for loading and exploring the FAIR^2 dataset on second primary colorectal cancer in cancer survivors using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview

Review available record sets, fields, and their IDs. All entities are referenced strictly using their `@id`.

We'll identify all available record sets, list their IDs, and for each, show available fields (columns) with their respective `@id`.

In [ ]:
# List all record sets with their @id
record_sets = []
for rs in dataset.record_sets:
    print(f"Record Set: {rs['@id']}")
    record_sets.append(rs['@id'])
    print("  Fields (columns):")
    for field in rs.get('field', []):
        # Field is either a dict or a string @id reference
        if isinstance(field, dict):
            print(f"    - {field['@id']}  (name: {field.get('name', '-')})")
        else:
            print(f"    - {field}")
    print()


## 3. Data Extraction

Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

We will extract each available record set (using its `@id`) and load the data into a pandas DataFrame.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded record set: {record_set_id}, num_records={len(records)}")

# Preview the first record set loaded
if record_sets:
    first_record_set_id = record_sets[0]
    print(f"First record set columns (@id): {dataframes[first_record_set_id].columns.tolist()}")
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filtering records, normalizing numeric fields, categorizing/grouping data.

Pick representative numeric and categorical fields (by their `@id`) from the previous overview.

*Note: Please adjust the field IDs below to match available ones in the dataset if needed.*

In [ ]:
# Select the record set and field @ids for EDA
record_set_id = record_sets[0] if record_sets else None

# List sample column @ids for guidance (adjust as needed)
if record_set_id is not None:
    print(f"Columns in {record_set_id}:")
    print(dataframes[record_set_id].columns.tolist())

    # EXAMPLE: Use @id for a numeric field. Replace with appropriate @id, e.g. 'https://api.app.sen.science/frontiers/7862866/diagnosis_interval_days',
    # and for grouping a categorical field, e.g. 'https://api.app.sen.science/frontiers/7862866/sex'
    # We'll try to pick two likely fields based on typical clinical datasets (please adjust after inspecting columns in the last cell):
    numeric_field = None
    group_field = None

    # Try auto-detect likely fields (You may want to edit IDs to match your needs later):
    for col in dataframes[record_set_id].columns:
        col_lo = col.lower()
        if (('age' in col_lo or 'interval' in col_lo or 'days' in col_lo) and numeric_field is None):
            numeric_field = col
        if (('sex' in col_lo or 'msi' in col_lo or 'site' in col_lo or 'location' in col_lo) and group_field is None):
            group_field = col

    if numeric_field and group_field:
        print(f"Using numeric field @id: {numeric_field}")
        print(f"Using group/categorical field @id: {group_field}")

        # Filter records with numeric_field > threshold, e.g. for 'diagnosis interval' field
        threshold = 10
        filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field].astype(float) > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (
            filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
        ) / filtered_df[numeric_field].astype(float).std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Group by category and report grouped mean
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field}, mean {numeric_field}:\n{grouped_df.head()}")

    else:
        print("Could not automatically detect both a numeric field and a group field. Please review the columns and select appropriate @id.")
else:
    print("No record set available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields using column `@id`. Below is an example of histogram and group comparison for the selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field for the filtered data
if record_set_id and numeric_field and group_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field].astype(float), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field].astype(float))
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()
else:
    print("No numeric and group fields available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated programmatic access to a FAIR-compliant clinical dataset via its Croissant schema and the `mlcroissant` library. We explored available record sets and their fields using unique `@id` values, loaded tabular data into pandas DataFrames, performed basic EDA (including filtering, normalization, and grouping), and visualized primary data distributions and category-level summaries.

**Key findings** will depend on the dataset's field values and the numeric/categorical columns selected for analysis. The approach ensures reproducibility, precise field referencing via `@id`, and harmonizes with FAIR data principles for responsible clinical data science.